# 📹 Home CCTV Smart Surveillance: Garbage Dumping & Person Detection using YOLO11
### End-to-End Computer Vision Pipeline for Real-World CCTV Security & Neighborhood Cleanliness
---
**📌 Project Overview:**
- **Domain:** Computer Vision, Smart Home Surveillance & Environmental Cleanliness
- **Source:** Outdoor Home Security Camera (Real-world CCTV stream & extracted frames)
- **Problem Statement:** Traditional CCTV systems record video passively without alerting homeowners when unauthorized littering or illegal garbage dumping occurs outside their home entrance. This project implements a real-time object detection system to automatically detect persons, garbage dumping activity (`Garbage_Person`), and stationary dumped garbage (`Garbage Stopped`).
- **Target Classes:**
  1. `Person` (Pedestrians, residents, visitors outside perimeter)
  2. `Garbage_Person` (Individual carrying, dumping, or handling garbage bags/trash)
  3. `Garbage Stopped` (Dumped garbage or trash bags left on the ground/street)
  4. `Custom-CCTV-Object-Detection` (General CCTV scene bounding / region of interest)
- **Model:** State-of-the-Art **Ultralytics YOLO11** (`yolo11n.pt`)
- **Annotation Platform:** Roboflow (`munawar-khan/custom-cctv-object-detection`)
- **Student / Author:** Munawar Khan
- **Submission Deadline:** 28th September

> 💡 **Google Colab Note:** Ensure GPU acceleration is enabled: Go to **Runtime -> Change runtime type -> T4 GPU**.

## 🛠️ Step 1: Install Dependencies & Verify Environment
We install the latest `ultralytics` package and inspect GPU acceleration availability.

In [ ]:
# Install Ultralytics and dependencies
!pip install -q ultralytics opencv-python matplotlib pyyaml pillow

import torch
import ultralytics
from ultralytics import YOLO
import os
import cv2
import glob
import zipfile
import yaml
import matplotlib.pyplot as plt
from PIL import Image

print(f"Ultralytics Version: {ultralytics.__version__}")
print(f"PyTorch Version:     {torch.__version__}")
print(f"CUDA Available:      {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device:   {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Running on CPU. Switch to T4 GPU in Colab Runtime for 10x faster training.")

## 📦 Step 2: Load & Prepare the Custom CCTV Dataset
The dataset was collected from our outdoor home CCTV camera, annotated on Roboflow, and exported in YOLO format.

The code below automatically detects and extracts any uploaded dataset `.zip` file, or detects the local folder, and configures `data.yaml` with correct absolute paths.

In [ ]:
# Automatically detect and extract dataset ZIP file if present
zip_files = glob.glob("*.zip")
dataset_dir = os.path.abspath("dataset")

if zip_files:
    target_zip = zip_files[0]
    print(f"📦 Found ZIP file: '{target_zip}'. Extracting to '{dataset_dir}'...")
    with zipfile.ZipFile(target_zip, 'r') as zip_ref:
        zip_ref.extractall(dataset_dir)
    print("✅ Extraction Complete!")
elif os.path.exists("train") and os.path.exists("data.yaml"):
    dataset_dir = os.path.abspath(".")
    print(f"✅ Using existing dataset folder at: {dataset_dir}")
else:
    print("⚠️ Please upload your dataset ZIP file (or place train/valid folders in workspace).")

# Locate and fix data.yaml to use robust absolute paths
yaml_candidates = glob.glob(os.path.join(dataset_dir, "**/data.yaml"), recursive=True) or glob.glob("**/data.yaml", recursive=True)
if yaml_candidates:
    data_yaml_path = yaml_candidates[0]
    base_dir = os.path.dirname(data_yaml_path)
    
    with open(data_yaml_path, 'r') as f:
        cfg = yaml.safe_load(f)
    
    # Ensure correct paths for YOLO training
    cfg['train'] = os.path.join(base_dir, 'train', 'images')
    cfg['val'] = os.path.join(base_dir, 'valid', 'images')
    cfg['test'] = os.path.join(base_dir, 'test', 'images')
    
    with open(data_yaml_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)
        
    print(f"\n📋 Configured data.yaml path: {data_yaml_path}")
    print(f"🏷️  Classes ({cfg.get('nc', 4)}): {cfg.get('names')}")
else:
    print("❌ Error: data.yaml could not be found.")

## 🔍 Step 3: Dataset Exploration & Sample Visualization
Let's inspect sample CCTV frames collected from the camera to view the surveillance scene and lighting conditions.

In [ ]:
# Discover sample train images
sample_images = glob.glob("**/train/images/*.jpg", recursive=True) + glob.glob("**/train/images/*.png", recursive=True)

print(f"Total Training Images Found: {len(sample_images)}")
if sample_images:
    plt.figure(figsize=(15, 7))
    for i, img_path in enumerate(sample_images[:4]):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.subplot(2, 2, i + 1)
        plt.imshow(img)
        plt.title(f"CCTV Frame {i+1}: {os.path.basename(img_path)[:25]}...")
        plt.axis('off')
    plt.suptitle("Sample Outdoor Home CCTV Training Frames", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 🧠 Step 4: YOLO11 Model Initialization
We use **Ultralytics YOLO11-Nano (`yolo11n.pt`)**:
- Lightweight (~2.6M parameters), making it fast and optimal for real-time edge CCTV streams.
- Features **C3k2** and **C2PSA** attention blocks to capture fine-grained objects (like trash bags and hand-carried garbage).
- Uses **Transfer Learning** pre-trained on the COCO dataset.

In [ ]:
# Load pre-trained YOLO11 Nano model
model = YOLO('yolo11n.pt')

print("✅ YOLO11 Pre-trained Weights Loaded Successfully!")
print(f"Base Model Architecture: {model.task.upper()} | Model Name: yolo11n.pt")

## 🚀 Step 5: Train YOLO11 on Custom CCTV Dataset
We fine-tune YOLO11 on our custom CCTV dataset using optimal hyperparameters:
- **epochs**: 30 (sufficient for convergence with pre-trained weights on 150+ augmented CCTV frames)
- **imgsz**: 640
- **batch**: 16 (or 8 on lower GPU memory)
- **patience**: 10 (early stopping)
- **plots**: True (automatically generates confusion matrix, PR curves, and loss curves)

In [ ]:
# Train YOLO11 on our custom CCTV dataset
train_results = model.train(
    data=data_yaml_path,
    epochs=30,
    imgsz=640,
    batch=16,
    patience=10,
    name='cctv_garbage_yolo11',
    save=True,
    plots=True,
    device=0 if torch.cuda.is_available() else 'cpu'
)

print("\n🎉 Training Complete! Model checkpoints and metrics saved.")

## 📊 Step 6: Model Evaluation & Performance Metrics
We evaluate the trained weights (`best.pt`) on the validation set to calculate **Precision**, **Recall**, and **mAP@50**.

In [ ]:
# Locate best model weights
best_weights = 'runs/detect/cctv_garbage_yolo11/weights/best.pt'
if not os.path.exists(best_weights):
    runs = glob.glob('runs/detect/**/weights/best.pt', recursive=True)
    if runs:
        best_weights = runs[-1]

print(f"Evaluating weights from: {best_weights}")
trained_model = YOLO(best_weights)

# Run validation
val_metrics = trained_model.val(data=data_yaml_path)

print("\n================ 📈 MODEL EVALUATION RESULTS ================")
print(f"Mean Average Precision @50 (mAP@50):    {val_metrics.box.map50 * 100:.2f}%")
print(f"Mean Average Precision @50-95:         {val_metrics.box.map * 100:.2f}%")
print(f"Overall Precision (P):                 {val_metrics.box.mp * 100:.2f}%")
print(f"Overall Recall (R):                    {val_metrics.box.mr * 100:.2f}%")
print("===========================================================")

### Display Evaluation Graphs (Loss Curves & Confusion Matrix)
Ultralytics automatically generates performance graphs in the run folder.

In [ ]:
# Display training curves and confusion matrix
results_dir = os.path.dirname(os.path.dirname(best_weights))

plots_to_show = [
    os.path.join(results_dir, 'results.png'),
    os.path.join(results_dir, 'confusion_matrix.png'),
    os.path.join(results_dir, 'PR_curve.png')
]

for p in plots_to_show:
    if os.path.exists(p):
        plt.figure(figsize=(10, 6))
        img = Image.open(p)
        plt.imshow(img)
        plt.title(os.path.basename(p), fontsize=12, fontweight='bold')
        plt.axis('off')
        plt.show()
    else:
        print(f"Graph not generated yet at: {p}")

## 🎯 Step 7: Model Predictions on Test CCTV Footage
We now test the trained model on unseen test images extracted from the CCTV camera.

In [ ]:
# Find test images
test_images = glob.glob("**/test/images/*.jpg", recursive=True) + glob.glob("**/test/images/*.png", recursive=True)
if not test_images:
    test_images = glob.glob("**/valid/images/*.jpg", recursive=True)[:4]

print(f"Found {len(test_images)} test images for evaluation.")

# Predict and visualize
for img_path in test_images[:4]:
    pred = trained_model.predict(
        source=img_path,
        conf=0.25,
        save=False,
        verbose=False
    )
    
    annotated_img = pred[0].plot()
    annotated_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
    
    # Detected labels
    boxes = pred[0].boxes
    labels = [trained_model.names[int(b.cls)] for b in boxes]
    
    print(f"\n📷 Test Image: {os.path.basename(img_path)}")
    print(f"   🚨 Detections ({len(labels)}): {labels}")
    
    plt.figure(figsize=(9, 6))
    plt.imshow(annotated_rgb)
    plt.title(f"Detections: {', '.join(set(labels)) if labels else 'No Detections'}", fontsize=11)
    plt.axis('off')
    plt.show()

## 🚨 Step 8: Real-Time CCTV Alert Logic (Dahua / Security Camera Deployment)
Below is the practical alerting function that triggers intelligent notifications:
- If `Garbage_Person` is detected -> **ALERT: Garbage Dumping Activity Detected!**
- If `Garbage Stopped` is detected -> **WARNING: Garbage Bag Left on Street!**
- If `Person` is detected -> **INFO: Person Detected Near Gate**

In [ ]:
def process_cctv_frame_with_alerts(frame, conf_threshold=0.30):
    """
    Takes a single CCTV frame, performs YOLO11 detection, and overlays smart alerts.
    """
    res = trained_model.predict(source=frame, conf=conf_threshold, verbose=False)
    annotated = res[0].plot()
    
    detected_classes = [trained_model.names[int(b.cls)].lower() for b in res[0].boxes]
    
    alert_y = 40
    # Priority 1: Garbage Dumping Activity
    if any("garbage_person" in c for c in detected_classes):
        cv2.rectangle(annotated, (10, alert_y - 30), (520, alert_y + 10), (0, 0, 200), -1)
        cv2.putText(annotated, "🚨 ALERT: Garbage Dumping Activity Detected!", (15, alert_y), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        alert_y += 45
        
    # Priority 2: Stationary Dumped Garbage on Ground
    if any("garbage stopped" in c for c in detected_classes):
        cv2.rectangle(annotated, (10, alert_y - 30), (500, alert_y + 10), (0, 140, 255), -1)
        cv2.putText(annotated, "⚠️ WARNING: Dumped Garbage / Bag on Street!", (15, alert_y), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        alert_y += 45
        
    # Priority 3: Person Near Gate
    if any(c == "person" for c in detected_classes):
        cv2.rectangle(annotated, (10, alert_y - 30), (400, alert_y + 10), (180, 100, 0), -1)
        cv2.putText(annotated, "🚶 Person Detected Near Gate", (15, alert_y), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
    return annotated

# Test alert overlay on a sample image
if test_images:
    sample_test = cv2.imread(test_images[0])
    alerted_frame = process_cctv_frame_with_alerts(sample_test)
    
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(alerted_frame, cv2.COLOR_BGR2RGB))
    plt.title("Live Smart Surveillance Alert Output Sample", fontsize=12, fontweight='bold')
    plt.axis('off')
    plt.show()

## 📋 Step 9: Assignment Deliverables & Summary

| Deliverable Item | Details / Links |
| :--- | :--- |
| **1. Notebook (.ipynb)** | `CCTV_Security_YOLO11_Detection.ipynb` (Full end-to-end pipeline) |
| **2. Dataset Link** | [Roboflow Dataset Link](https://app.roboflow.com/munawar-khan/custom-cctv-object-detection/1) |
| **3. Project Report** | `PROJECT_REPORT.md` (Detailed academic & technical summary) |
| **4. Problem Statement** | Real-World Home CCTV Garbage Dumping & Person Detection |
| **5. Model Used** | Ultralytics YOLO11 (`yolo11n.pt`) |